In [1]:
import pandas as pd
import numpy as np
import importlib
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import src.feature_engineering

In [2]:
BASE_DIR = Path("..")
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_INTERIM = BASE_DIR / "data" / "interim"
DATA_PROCESSED = BASE_DIR / "data" / "processed"

In [3]:
train_train = pd.read_csv(DATA_RAW/"train.csv")
train_test = pd.read_csv(DATA_RAW/"train_v2.csv")

In [6]:
members = pd.read_csv(DATA_RAW/"members_v3.csv")

In [4]:
transactions = pd.read_csv(DATA_INTERIM/"transactions.csv")

In [7]:
transactions.head()

,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,20150930,20151101,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,20150930,20151031,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,20150930,20160427,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,20150930,20151128,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,20150930,20151121,0


In [8]:
importlib.reload(src.feature_engineering)

<module 'src.feature_engineering' from '/Volumes/Kingston/Alesia/vs_projects/kkbox_churn/src/feature_engineering.py'>

In [7]:
train_features = src.feature_engineering.build_features(
    train_train,
    members, 
    transactions, 
    "2017-01-31", 
    path_to_user_logs=DATA_INTERIM/"user_logs.parquet"
)

Aggregate user_logs...
Aggregate transactions...


In [ ]:
test_features = src.feature_engineering.build_features(
    train_test,
    members, 
    transactions, 
    "2017-02-28", 
    path_to_user_logs=DATA_INTERIM/"user_logs.parquet"
)

In [19]:
train_train.shape, train_test.shape

((992931, 2), (970960, 2))

In [20]:
train_features.shape, test_features.shape

((992931, 37), (970960, 37))

In [21]:
train_features = src.feature_engineering.features_engineering(
    train_features,
    "2017-01-31"
)

In [23]:
test_features = src.feature_engineering.features_engineering(
    test_features,
    "2017-02-28"
)

In [25]:
print(len(train_features.columns) == len(test_features.columns))

True


In [58]:
print(all(test_features.dtypes == train_features.dtypes))

True


In [26]:
train_features.isna().sum()

msno                                 0
is_churn                             0
city                                 0
bd                              603791
gender                               0
registered_via                       0
payment_method_most_frequent         0
plan_days_mean                       0
plan_list_price_median               0
actual_amount_paid_median            0
discount_amount_mean                 0
is_discount_rate                     0
auto_renew_rate                      0
total_transactions                   0
is_cancel_rate                       0
transaction_history_days             0
total_num_25                         0
total_num_50                         0
total_num_75                         0
total_num_985                        0
total_num_100                        0
total_num_unq                        0
total_secs                           0
active_days                          0
total_plays                          0
skip_rate                

In [27]:
test_features.isna().sum()

msno                                 0
is_churn                             0
city                                 0
bd                              584245
gender                               0
registered_via                       0
payment_method_most_frequent         0
plan_days_mean                       0
plan_list_price_median               0
actual_amount_paid_median            0
discount_amount_mean                 0
is_discount_rate                     0
auto_renew_rate                      0
total_transactions                   0
is_cancel_rate                       0
transaction_history_days             0
total_num_25                         0
total_num_50                         0
total_num_75                         0
total_num_985                        0
total_num_100                        0
total_num_unq                        0
total_secs                           0
active_days                          0
total_plays                          0
skip_rate                

In [30]:
day_cols = [
    "plan_days_mean",
    "transaction_history_days",
    "active_days",
    "days_since_last_activity",
    "days_since_last_transaction"
]

train_features[day_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
plan_days_mean,992931.0,33.640246,32.950421,0.0,28.846154,30.0,30.0,450.0
transaction_history_days,992931.0,464.539116,248.060667,0.0,246.000000,488.0,729.0,761.0
active_days,992931.0,102.955399,97.705202,0.0,14.000000,74.0,176.0,342.0
days_since_last_activity,992931.0,34.823310,85.586352,-1.0,0.000000,5.0,28.0,761.0
days_since_last_transaction,992931.0,19.249929,41.467191,-1.0,5.000000,14.0,23.0,760.0


In [31]:
test_features[day_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
plan_days_mean,970960.0,34.051306,34.279500,0.0,28.888889,30.0,30.0,450.0
transaction_history_days,970960.0,482.518891,258.997036,0.0,249.000000,516.0,758.0,789.0
active_days,970960.0,106.828112,101.123960,0.0,15.000000,79.0,179.0,352.0
days_since_last_activity,970960.0,35.693702,88.556300,-1.0,0.000000,5.0,29.0,789.0
days_since_last_transaction,970960.0,17.351723,43.742320,-1.0,2.000000,11.0,20.0,788.0


In [34]:
rate_cols = [
    "auto_renew_rate",
    "is_discount_rate",
    "is_cancel_rate",
    "skip_rate",
    "completed_rate"
]

for name, df in {"train_features": train_features, "test_features": test_features}.items():
    print(name)
    for col in rate_cols:
        print(
            col,
            "below 0:", (df[col] < 0).sum(),
            "above 1:", (df[col] > 1).sum()
        )

train_features
auto_renew_rate below 0: 0 above 1: 0
is_discount_rate below 0: 0 above 1: 0
is_cancel_rate below 0: 0 above 1: 0
skip_rate below 0: 0 above 1: 0
completed_rate below 0: 0 above 1: 0
test_features
auto_renew_rate below 0: 0 above 1: 0
is_discount_rate below 0: 0 above 1: 0
is_cancel_rate below 0: 0 above 1: 0
skip_rate below 0: 0 above 1: 0
completed_rate below 0: 0 above 1: 0


In [35]:
numeric_cols = train_features.select_dtypes(
    include="number"
).columns

summary = train_features[numeric_cols].describe().T

summary["missing"] = train_features[numeric_cols].isna().sum()
summary["negative"] = (
    train_features[numeric_cols] < 0
).sum()
summary["zero"] = (
    train_features[numeric_cols] == 0
).sum()

summary[
    [
        "min",
        "max",
        "mean",
        "50%",
        "std",
        "missing",
        "negative",
        "zero"
    ]
]

,min,max,mean,50%,std,missing,negative,zero
is_churn,0.0,1.0,0.063923,0.0,0.244616,0,0,929460
bd,1.0,100.0,29.971141,28.0,8.904479,603791,0,0
plan_days_mean,0.0,450.0,33.640246,30.0,32.950421,0,0,2097
plan_list_price_median,0.0,2000.0,145.89157,149.0,139.898212,0,0,3449
actual_amount_paid_median,0.0,2000.0,145.627926,149.0,140.463627,0,0,5012
discount_amount_mean,-1788.0,177.666667,-3.045901,0.0,14.914253,0,280960,645013
is_discount_rate,0.0,1.0,0.015507,0.0,0.068494,0,0,907171
auto_renew_rate,0.0,1.0,0.872861,1.0,0.313714,0,0,101245
total_transactions,0.0,244.0,15.21919,16.0,8.354461,0,0,2095
is_cancel_rate,0.0,1.0,0.013855,0.0,0.036177,0,0,792244


In [36]:
Q1 = train_features["total_secs"].quantile(0.25)
Q3 = train_features["total_secs"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = train_features[
    (train_features["total_secs"] < lower) | (train_features["total_secs"] > upper)
]

print("Outliers:", len(outliers))

Outliers: 67029


In [37]:
print(len(train_features[train_features["total_secs"] < lower]))

0


In [38]:
(train_features[day_cols] < 0).sum()

plan_days_mean                      0
transaction_history_days            0
active_days                         0
days_since_last_activity       169821
days_since_last_transaction      2095
dtype: int64

In [40]:
(train_features["total_secs"] >= 9e15).sum()

np.int64(0)

In [50]:
train_features["total_secs"].nlargest(20)

286758    2.774147e+07
278297    2.748192e+07
584196    2.682451e+07
250000    2.651249e+07
665391    2.591425e+07
533821    2.588945e+07
236975    2.574521e+07
854748    2.572863e+07
942955    2.564513e+07
305271    2.551950e+07
864932    2.532972e+07
74175     2.526604e+07
808016    2.518393e+07
826804    2.512034e+07
527971    2.508717e+07
962259    2.487385e+07
518744    2.477413e+07
932842    2.463560e+07
581383    2.459138e+07
826526    2.412603e+07
Name: total_secs, dtype: float64

In [41]:
train_features["msno"].duplicated().sum()

np.int64(0)

In [42]:
train_features[
    (train_features["total_transactions"] == 0) &
    (train_features["actual_amount_paid_median"] > 0)
].shape

(0, 39)

In [43]:
train_features[
    (train_features["total_transactions"] == 0) &
    (train_features["discount_amount_mean"] > 0)
].shape

(0, 39)

In [44]:
numeric_cols = test_features.select_dtypes(
    include="number"
).columns

summary = test_features[numeric_cols].describe().T

summary["missing"] = test_features[numeric_cols].isna().sum()
summary["negative"] = (
    test_features[numeric_cols] < 0
).sum()
summary["zero"] = (
    test_features[numeric_cols] == 0
).sum()

summary[
    [
        "min",
        "max",
        "mean",
        "50%",
        "std",
        "missing",
        "negative",
        "zero"
    ]
]

,min,max,mean,50%,std,missing,negative,zero
is_churn,0.0,1.0,0.089942,0.0,0.286099,0,0,883630
bd,1.0,100.0,29.904216,28.0,8.920391,584245,0,0
plan_days_mean,0.0,450.0,34.051306,30.0,34.2795,0,0,2526
plan_list_price_median,0.0,2000.0,147.570869,149.0,145.608924,0,0,3365
actual_amount_paid_median,0.0,2000.0,147.227777,149.0,146.003134,0,0,4929
discount_amount_mean,-1788.0,174.75,-2.848006,0.0,14.229725,0,270493,635898
is_discount_rate,0.0,1.0,0.014817,0.0,0.067087,0,0,888342
auto_renew_rate,0.0,1.0,0.867536,1.0,0.319158,0,0,103564
total_transactions,0.0,244.0,15.748153,16.0,8.707882,0,0,2524
is_cancel_rate,0.0,1.0,0.013932,0.0,0.035999,0,0,769567


In [45]:
Q1 = test_features["total_secs"].quantile(0.25)
Q3 = test_features["total_secs"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = test_features[
    (test_features["total_secs"] < lower) | (test_features["total_secs"] > upper)
]

print("Outliers:", len(outliers))

Outliers: 64977


In [46]:
print(len(test_features[test_features["total_secs"] < lower]))

0


In [48]:
(test_features["total_secs"] >= 9e15).sum()

np.int64(0)

In [49]:
test_features["total_secs"].nlargest(20)

636339    2.923582e+07
596244    2.868158e+07
387760    2.830636e+07
317026    2.734974e+07
92428     2.724401e+07
789897    2.695233e+07
800457    2.685199e+07
122618    2.674314e+07
583772    2.669302e+07
384187    2.650350e+07
535320    2.639431e+07
594513    2.624900e+07
37433     2.581567e+07
931570    2.581067e+07
332022    2.575242e+07
710308    2.550624e+07
593826    2.526960e+07
623226    2.526604e+07
543252    2.514999e+07
216368    2.503293e+07
Name: total_secs, dtype: float64

In [51]:
test_features["msno"].duplicated().sum()

np.int64(0)

In [52]:
test_features[
    (test_features["total_transactions"] == 0) &
    (test_features["actual_amount_paid_median"] > 0)
].shape

(0, 39)

In [53]:
test_features[
    (train_features["total_transactions"] == 0) &
    (train_features["discount_amount_mean"] > 0)
].shape

/var/folders/38/xt3g60vn0q53byw9v6c9dks80000gn/T/ipykernel_3184/1191826252.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  test_features[


(0, 39)

In [56]:
train_features.to_csv(
    DATA_PROCESSED/"train_features.csv",
    index=False
)

test_features.to_csv(
    DATA_PROCESSED/"test_features.csv",
    index=False
)

In [57]:
import os

print(os.path.getsize(DATA_PROCESSED/"train_features.csv") / 1024**2, "MB")
print(os.path.getsize(DATA_PROCESSED/"test_features.csv") / 1024**2, "MB")

283.3541421890259 MB
278.502158164978 MB
